In [1]:
!pip install -q transformers datasets accelerate scikit-learn pandas evaluate
!pip install -q --upgrade kaggle

import torch
print(torch.__version__)
print("GPU available:", torch.cuda.is_available())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.2/126.2 kB 3.9 MB/s eta 0:00:00
2.11.0+cu128
GPU available: True


In [2]:
import pandas as pd

df = pd.read_csv("cefr_leveled_texts.csv")
print(df.shape)
print(df.columns.tolist())
print(df.head())

(1494, 2)
['text', 'label']
                                                text label
0  Hi!\nI've been meaning to write for ages and f...    B2
1  ﻿It was not so much how hard people found the ...    B2
2  Keith recently came back from a trip to Chicag...    B2
3  The Griffith Observatory is a planetarium, and...    B2
4  -LRB- The Hollywood Reporter -RRB- It's offici...    B2


In [3]:
print(df["label"].value_counts())

label
A1    288
B2    286
A2    272
C1    241
B1    205
C2    202
Name: count, dtype: int64


In [4]:
cefr_order = ["A1", "A2", "B1", "B2", "C1", "C2"]
label2id = {lvl: i for i, lvl in enumerate(cefr_order)}
id2label = {i: lvl for lvl, i in label2id.items()}

df = df.rename(columns={"label": "cefr_level"})   # keep the original string around
df["label"] = df["cefr_level"].map(label2id)

print(df["label"].isna().sum())   # must be 0
print(df[["cefr_level", "label"]].head())

0
  cefr_level  label
0         B2      3
1         B2      3
2         B2      3
3         B2      3
4         B2      3


In [5]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report

train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df["label"]
)

vectorizer = TfidfVectorizer(max_features=5000)
X_train = vectorizer.fit_transform(train_df["text"])
X_test = vectorizer.transform(test_df["text"])

baseline = LogisticRegression(max_iter=1000, class_weight="balanced").fit(X_train, train_df["label"])
pred = baseline.predict(X_test)

baseline_f1 = f1_score(test_df["label"], pred, average="macro")
print("Baseline macro-F1:", baseline_f1)
print(classification_report(test_df["label"], pred, target_names=cefr_order))

Baseline macro-F1: 0.5850121647877499
              precision    recall  f1-score   support

          A1       0.75      0.83      0.79        58
          A2       0.72      0.71      0.72        55
          B1       0.50      0.34      0.41        41
          B2       0.51      0.44      0.47        57
          C1       0.43      0.62      0.51        48
          C2       0.68      0.57      0.62        40

    accuracy                           0.60       299
   macro avg       0.60      0.59      0.59       299
weighted avg       0.60      0.60      0.59       299



In [6]:
from datasets import Dataset
from transformers import AutoTokenizer

MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_ds = Dataset.from_pandas(train_df[["text", "label"]].reset_index(drop=True))
test_ds = Dataset.from_pandas(test_df[["text", "label"]].reset_index(drop=True))

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=256)

train_ds = train_ds.map(tokenize, batched=True)
test_ds = test_ds.map(tokenize, batched=True)
train_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/1195 [00:00<?, ? examples/s]

Map:   0%|          | 0/299 [00:00<?, ? examples/s]

In [7]:
example = train_ds[0]
print(tokenizer.decode(example["input_ids"], skip_special_tokens=True)[:300])
print("Original:", train_df["text"].iloc[0][:300])
print("Label:", id2label[int(example["label"])])

they call it the richie rich club and it is about to get even richer. india ’ s wealthiest will quadruple their net worth between now and 2018, a report says, with hundreds of thousands of new entrepreneurs and inheritors becoming multimillionaires. the survey, based on interviews with 150 ultra - h
Original: ﻿They call it the Richie Rich Club and it is about to get even richer. India’s wealthiest will quadruple their net worth between now and 2018, a report says, with hundreds of thousands of new entrepreneurs and inheritors becoming multimillionaires. The survey, based on interviews with 150 ultra-high
Label: C1


In [9]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import f1_score
import numpy as np
import datasets.config
datasets.config.TORCHVISION_AVAILABLE = False

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(cefr_order), id2label=id2label, label2id=label2id
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"macro_f1": f1_score(labels, preds, average="macro")}

args = TrainingArguments(
    output_dir="./cefr_classifier_model",
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=20,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=test_ds,
    compute_metrics=compute_metrics,
)
trainer.train()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Macro F1
1,1.311898,1.129620,0.562211
2,0.986018,0.916511,0.581133
3,0.846670,0.860221,0.645297
4,0.744480,0.841498,0.644698


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=300, training_loss=1.0271975199381511, metrics={'train_runtime': 150.6434, 'train_samples_per_second': 31.731, 'train_steps_per_second': 1.991, 'total_flos': 316619667025920.0, 'train_loss': 1.0271975199381511, 'epoch': 4.0})

In [10]:
from sklearn.metrics import confusion_matrix, classification_report, f1_score
import numpy as np

preds_output = trainer.predict(test_ds)
preds = np.argmax(preds_output.predictions, axis=-1)
labels = preds_output.label_ids

print("Transformer macro-F1:", f1_score(labels, preds, average="macro"))
print("Baseline macro-F1 was:", baseline_f1)
print(classification_report(labels, preds, target_names=cefr_order))
print(confusion_matrix(labels, preds))

Transformer macro-F1: 0.6452967349251798
Baseline macro-F1 was: 0.5850121647877499
              precision    recall  f1-score   support

          A1       0.77      0.91      0.83        58
          A2       0.70      0.73      0.71        55
          B1       0.72      0.44      0.55        41
          B2       0.49      0.61      0.55        57
          C1       0.53      0.44      0.48        48
          C2       0.78      0.72      0.75        40

    accuracy                           0.66       299
   macro avg       0.67      0.64      0.65       299
weighted avg       0.66      0.66      0.65       299

[[53  5  0  0  0  0]
 [15 40  0  0  0  0]
 [ 1  7 18 14  1  0]
 [ 0  5  6 35  9  2]
 [ 0  0  1 20 21  6]
 [ 0  0  0  2  9 29]]
